# DeepSORT Modernization Project

Final Colab notebook for submission.

This notebook reproduces the main pipeline for the modified DeepSORT project:

1. clone the GitHub repository,
2. install dependencies,
3. download the required MOT-style dataset,
4. run original DeepSORT baseline,
5. run YOLO11n person detection,
6. extract OSNet x0.5 ReID features,
7. run the final best DeepSORT configuration,
8. generate 12 overlay videos,
9. display original and best overlays for visual comparison,
10. optionally save generated outputs to Google Drive.

Final selected configuration:

- detector: YOLO11n;
- detector confidence threshold: 0.40;
- detector IoU threshold: 0.60;
- ReID model: OSNet x0.5;
- DeepSORT `max_cosine_distance`: 0.30;
- `nn_budget`: 100.

The report with full numerical results is available in `report/report.md`.


## 1. Clean setup: clone repository


In [2]:


%cd /content

!rm -rf /content/deepsort-modern-project

!git clone https://github.com/Dokirma/deepsort-modern-project.git /content/deepsort-modern-project

%cd /content/deepsort-modern-project

!pwd
!ls
!test -f deep_sort_app.py && echo "deep_sort_app.py OK"
!test -f src/reid/osnet_extractor.py && echo "osnet_extractor.py OK"
!test -f src/visualization/make_overlay.py && echo "make_overlay.py OK"

/content
Cloning into '/content/deepsort-modern-project'...
remote: Enumerating objects: 385, done.
remote: Counting objects: 100% (385/385), done.
remote: Compressing objects: 100% (172/172), done.
remote: Total 385 (delta 206), reused 384 (delta 205), pack-reused 0 (from 0)
Receiving objects: 100% (385/385), 124.58 KiB | 1.05 MiB/s, done.
Resolving deltas: 100% (206/206), done.
/content/deepsort-modern-project
/content/deepsort-modern-project
application_util  evaluate_motchallenge.py  README.md		  src
artifacts	  generate_videos.py	    report		  tools
configs		  LICENSE		    requirements-gpu.txt
deep_sort	  notebooks		    requirements.txt
deep_sort_app.py  outputs		    show_results.py
deep_sort_app.py OK
osnet_extractor.py OK
make_overlay.py OK


## 2. Install dependencies


In [8]:
%cd /content/deepsort-modern-project

!pip install -q -r requirements.txt
!pip install -q ultralytics torchreid gdown

/content/deepsort-modern-project


## 3. Download and extract dataset


In [9]:
%cd /content/deepsort-modern-project

!python src/data/download_data.py \
  --file-id 1ujjjDlQZ6eEfdfWqJx-L_pgbJkSqRkU8 \
  --output data/raw/dlcv-final-project-videos.tar.gz \
  --extract-to data/mot

!find data/mot/videos -maxdepth 1 -type d | sort

/content/deepsort-modern-project
Downloading...
From (original): https://drive.google.com/uc?id=1ujjjDlQZ6eEfdfWqJx-L_pgbJkSqRkU8
From (redirected): https://drive.google.com/uc?id=1ujjjDlQZ6eEfdfWqJx-L_pgbJkSqRkU8&confirm=t&uuid=6c0a9ac8-f094-43b2-8441-45d6f06ecc8f
To: /content/deepsort-modern-project/data/raw/dlcv-final-project-videos.tar.gz
100% 364M/364M [00:05<00:00, 67.3MB/s]
Extracting dataset...
/content/deepsort-modern-project/src/data/download_data.py:16: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  archive.extractall(extract_to)
Done.
data/mot/videos
data/mot/videos/KITTI-17
data/mot/videos/MOT16-09
data/mot/videos/MOT16-11
data/mot/videos/PETS09-S2L1
data/mot/videos/TUD-Campus
data/mot/videos/TUD-Stadtmitte


## 4. Check project scripts


In [10]:
%cd /content/deepsort-modern-project

!ls src/tracking
!ls src/detectors
!ls src/reid
!ls src/visualization

/content/deepsort-modern-project
run_deepsort_from_detections_all.py  run_original_deepsort_all.py
run_deepsort_from_reid_all.py	     simple_iou_tracker.py
run_yolo_all.py  yolo_detector.py
osnet_extractor.py  run_osnet_features_all.py
make_overlay.py


## 5. Check dataset folders


In [12]:
%cd /content/deepsort-modern-project

!find data/mot/videos -maxdepth 1 -type d | sort || echo "no data yet"

/content/deepsort-modern-project
data/mot/videos
data/mot/videos/KITTI-17
data/mot/videos/MOT16-09
data/mot/videos/MOT16-11
data/mot/videos/PETS09-S2L1
data/mot/videos/TUD-Campus
data/mot/videos/TUD-Stadtmitte


## 6. Run original DeepSORT baseline on all six sequences


In [13]:
%cd /content/deepsort-modern-project

!python src/tracking/run_original_deepsort_all.py

!echo "Original tracks:"
!ls -lh outputs/tracks/original_deepsort
!wc -l outputs/tracks/original_deepsort/*.txt

/content/deepsort-modern-project
Running original DeepSORT baseline for KITTI-17
/usr/bin/python3 src/data/convert_det_to_npy.py --det-txt data/mot/videos/KITTI-17/det/det.txt --output-npy outputs/detections/KITTI-17_pseudo_features.npy
Saved: outputs/detections/KITTI-17_pseudo_features.npy
Shape: (552, 138)
/usr/bin/python3 deep_sort_app.py --sequence_dir data/mot/videos/KITTI-17 --detection_file outputs/detections/KITTI-17_pseudo_features.npy --output_file outputs/tracks/original_deepsort/KITTI-17.txt --min_confidence 0 --min_detection_height 0 --nms_max_overlap 1.0 --max_cosine_distance 0.2 --nn_budget 100 --display False
Processing frame 00001
Processing frame 00002
Processing frame 00003
Processing frame 00004
Processing frame 00005
Processing frame 00006
Processing frame 00007
Processing frame 00008
Processing frame 00009
Processing frame 00010
Processing frame 00011
Processing frame 00012
Processing frame 00013
Processing frame 00014
Processing frame 00015
Processing frame 00016

## 7. Run YOLO11n detections on all six sequences


In [14]:
%cd /content/deepsort-modern-project

!python src/detectors/run_yolo_all.py \
  --model yolo11n.pt \
  --conf 0.40 \
  --iou 0.60 \
  --imgsz 640 \
  --name yolo11n_conf040

!echo "YOLO detections:"
!ls -lh outputs/detections/yolo11n_conf040

/content/deepsort-modern-project
Running yolo11n.pt on KITTI-17
/usr/bin/python3 src/detectors/yolo_detector.py --sequence-dir data/mot/videos/KITTI-17 --output-file outputs/detections/yolo11n_conf040/KITTI-17.txt --model yolo11n.pt --conf 0.4 --iou 0.6 --imgsz 640
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Model: yolo11n.pt
Sequence: data/mot/videos/KITTI-17
Frames: 145
Detections: 617
Total time: 3.42 sec
FPS: 42.40
Saved: outputs/detections/yolo11n_conf040/KITTI-17.txt
Running yolo11n.pt on MOT16-09
/usr/bin/python3 src/detectors/yolo_detector.py --sequence-dir data/mot/videos/MOT16-09 --output-file outputs/detections/yolo11n_conf040/MOT16-09.txt --model yolo11n.pt --conf 0.4 --iou 0.6 --imgsz 640
Model: yolo11n.pt
Sequ

## 8. Extract OSNet x0.5 ReID features


In [15]:
%cd /content/deepsort-modern-project

!python src/reid/run_osnet_features_all.py \
  --det-dir outputs/detections/yolo11n_conf040 \
  --output-dir outputs/detections_reid/yolo11n_osnet_x0_5 \
  --model osnet_x0_5 \
  --batch-size 32

!echo "OSNet x0.5 features:"
!ls -lh outputs/detections_reid/yolo11n_osnet_x0_5

/content/deepsort-modern-project
Extracting ReID features for KITTI-17
/usr/bin/python3 src/reid/osnet_extractor.py --sequence-dir data/mot/videos/KITTI-17 --det-file outputs/detections/yolo11n_conf040/KITTI-17.txt --output-npy outputs/detections_reid/yolo11n_osnet_x0_5/KITTI-17.npy --model osnet_x0_5 --batch-size 32
/usr/local/lib/python3.12/dist-packages/torchreid/reid/metrics/rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(
2026-06-26 14:45:29.560347: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Downloading...
From: https://drive.google.com/uc?id=16DGLbZukvVYgINws8u8deSaOqjybZ83i
To: /root/.cache/torch/checkpoints/osnet_x0_5_imagenet.pth
100% 4.71M/4.71M

## 9. Run final best DeepSORT configuration


In [16]:
%cd /content/deepsort-modern-project

!python src/tracking/run_deepsort_from_reid_all.py \
  --reid-dir outputs/detections_reid/yolo11n_osnet_x0_5 \
  --tracker-name deepsort_yolo11n_osnet_x0_5_cos030 \
  --max-cosine-distance 0.30 \
  --nn-budget 100

!echo "Best tracks:"
!ls -lh outputs/tracks/deepsort_yolo11n_osnet_x0_5_cos030
!wc -l outputs/tracks/deepsort_yolo11n_osnet_x0_5_cos030/*.txt

/content/deepsort-modern-project
Running DeepSORT with ReID features for KITTI-17
/usr/bin/python3 deep_sort_app.py --sequence_dir data/mot/videos/KITTI-17 --detection_file outputs/detections_reid/yolo11n_osnet_x0_5/KITTI-17.npy --output_file outputs/tracks/deepsort_yolo11n_osnet_x0_5_cos030/KITTI-17.txt --min_confidence 0 --min_detection_height 0 --nms_max_overlap 1.0 --max_cosine_distance 0.30 --nn_budget 100 --display False
Processing frame 00001
Processing frame 00002
Processing frame 00003
Processing frame 00004
Processing frame 00005
Processing frame 00006
Processing frame 00007
Processing frame 00008
Processing frame 00009
Processing frame 00010
Processing frame 00011
Processing frame 00012
Processing frame 00013
Processing frame 00014
Processing frame 00015
Processing frame 00016
Processing frame 00017
Processing frame 00018
Processing frame 00019
Processing frame 00020
Processing frame 00021
Processing frame 00022
Processing frame 00023
Processing frame 00024
Processing frame 

## 10. Generate 12 overlay videos


In [17]:
%cd /content/deepsort-modern-project

import os
import subprocess

sequences = [
    "KITTI-17",
    "MOT16-09",
    "MOT16-11",
    "PETS09-S2L1",
    "TUD-Campus",
    "TUD-Stadtmitte",
]

original_video_dir = "outputs/videos/original_deepsort"
best_video_dir = "outputs/videos/best_yolo11n_osnet_x0_5"

os.makedirs(original_video_dir, exist_ok=True)
os.makedirs(best_video_dir, exist_ok=True)

for seq in sequences:
    print("=" * 80)
    print(f"Creating ORIGINAL overlay for {seq}")

    subprocess.run([
        "python", "src/visualization/make_overlay.py",
        "--sequence-dir", f"data/mot/videos/{seq}",
        "--tracks-file", f"outputs/tracks/original_deepsort/{seq}.txt",
        "--output-video", f"{original_video_dir}/{seq}_original_deepsort.mp4",
    ], check=True)

    print(f"Creating BEST overlay for {seq}")

    subprocess.run([
        "python", "src/visualization/make_overlay.py",
        "--sequence-dir", f"data/mot/videos/{seq}",
        "--tracks-file", f"outputs/tracks/deepsort_yolo11n_osnet_x0_5_cos030/{seq}.txt",
        "--output-video", f"{best_video_dir}/{seq}_best_yolo11n_osnet_x0_5.mp4",
    ], check=True)

print("Done. Created 12 overlay videos.")

/content/deepsort-modern-project
Creating ORIGINAL overlay for KITTI-17
Creating BEST overlay for KITTI-17
Creating ORIGINAL overlay for MOT16-09
Creating BEST overlay for MOT16-09
Creating ORIGINAL overlay for MOT16-11
Creating BEST overlay for MOT16-11
Creating ORIGINAL overlay for PETS09-S2L1
Creating BEST overlay for PETS09-S2L1
Creating ORIGINAL overlay for TUD-Campus
Creating BEST overlay for TUD-Campus
Creating ORIGINAL overlay for TUD-Stadtmitte
Creating BEST overlay for TUD-Stadtmitte
Done. Created 12 overlay videos.


## 11. Verify 12 original/best overlay videos


In [18]:
%cd /content/deepsort-modern-project

!echo "ORIGINAL OVERLAYS:"
!ls -lh outputs/videos/original_deepsort/*.mp4

!echo ""
!echo "BEST OVERLAYS:"
!ls -lh outputs/videos/best_yolo11n_osnet_x0_5/*.mp4

!echo ""
!echo "COUNT:"
!find outputs/videos/original_deepsort outputs/videos/best_yolo11n_osnet_x0_5 -name "*.mp4" | wc -l

/content/deepsort-modern-project
ORIGINAL OVERLAYS:
-rw-r--r-- 1 root root 6.9M Jun 26 14:48 outputs/videos/original_deepsort/KITTI-17_original_deepsort.mp4
-rw-r--r-- 1 root root  62M Jun 26 14:49 outputs/videos/original_deepsort/MOT16-09_original_deepsort.mp4
-rw-r--r-- 1 root root 106M Jun 26 14:49 outputs/videos/original_deepsort/MOT16-11_original_deepsort.mp4
-rw-r--r-- 1 root root  27M Jun 26 14:50 outputs/videos/original_deepsort/PETS09-S2L1_original_deepsort.mp4
-rw-r--r-- 1 root root 1.8M Jun 26 14:50 outputs/videos/original_deepsort/TUD-Campus_original_deepsort.mp4
-rw-r--r-- 1 root root 4.2M Jun 26 14:50 outputs/videos/original_deepsort/TUD-Stadtmitte_original_deepsort.mp4

BEST OVERLAYS:
-rw-r--r-- 1 root root 7.0M Jun 26 14:48 outputs/videos/best_yolo11n_osnet_x0_5/KITTI-17_best_yolo11n_osnet_x0_5.mp4
-rw-r--r-- 1 root root  60M Jun 26 14:49 outputs/videos/best_yolo11n_osnet_x0_5/MOT16-09_best_yolo11n_osnet_x0_5.mp4
-rw-r--r-- 1 root root  99M Jun 26 14:50 outputs/videos/b

## 12. Links to overlay videos


In [20]:
%cd /content/deepsort-modern-project

from IPython.display import HTML, display

sequences = [
    "KITTI-17",
    "MOT16-09",
    "MOT16-11",
    "PETS09-S2L1",
    "TUD-Campus",
    "TUD-Stadtmitte",
]

rows = []

for seq in sequences:
    original_path = f"outputs/videos/original_deepsort/{seq}_original_deepsort.mp4"
    best_path = f"outputs/videos/best_yolo11n_osnet_x0_5/{seq}_best_yolo11n_osnet_x0_5.mp4"

    rows.append(f"""
    <tr>
        <td>{seq}</td>
        <td><a href="{original_path}" target="_blank">Original overlay</a></td>
        <td><a href="{best_path}" target="_blank">Best overlay</a></td>
    </tr>
    """)

html = f"""
<h3>Overlay videos for all six sequences</h3>
<table border="1" style="border-collapse: collapse;">
    <tr>
        <th>Sequence</th>
        <th>Original DeepSORT</th>
        <th>Best modified DeepSORT</th>
    </tr>
    {''.join(rows)}
</table>
"""

display(HTML(html))

/content/deepsort-modern-project


Sequence,Original DeepSORT,Best modified DeepSORT
KITTI-17,Original overlay,Best overlay
MOT16-09,Original overlay,Best overlay
MOT16-11,Original overlay,Best overlay
PETS09-S2L1,Original overlay,Best overlay
TUD-Campus,Original overlay,Best overlay
TUD-Stadtmitte,Original overlay,Best overlay


## 13. Check video duration


In [24]:
%cd /content/deepsort-modern-project

!ffprobe -v error \
  -show_entries format=duration,size \
  -of default=noprint_wrappers=1 \
  outputs/videos/original_deepsort/TUD-Campus_original_deepsort.mp4

!ffprobe -v error \
  -show_entries format=duration,size \
  -of default=noprint_wrappers=1 \
  outputs/videos/best_yolo11n_osnet_x0_5/TUD-Campus_best_yolo11n_osnet_x0_5.mp4

/content/deepsort-modern-project
duration=2.840000
size=1846855
duration=2.840000
size=1883964


## 14. Convert overlay videos to browser-compatible H.264


In [25]:
%cd /content/deepsort-modern-project

!mkdir -p outputs/videos_web/original_deepsort
!mkdir -p outputs/videos_web/best_yolo11n_osnet_x0_5

!for f in outputs/videos/original_deepsort/*.mp4; do \
  base=$(basename "$f"); \
  ffmpeg -y -i "$f" \
    -vcodec libx264 -pix_fmt yuv420p -movflags +faststart \
    -preset veryfast -crf 28 \
    "outputs/videos_web/original_deepsort/$base"; \
done

!for f in outputs/videos/best_yolo11n_osnet_x0_5/*.mp4; do \
  base=$(basename "$f"); \
  ffmpeg -y -i "$f" \
    -vcodec libx264 -pix_fmt yuv420p -movflags +faststart \
    -preset veryfast -crf 28 \
    "outputs/videos_web/best_yolo11n_osnet_x0_5/$base"; \
done

/content/deepsort-modern-project
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable

## 15. Verify browser-compatible videos


In [26]:
%cd /content/deepsort-modern-project

!echo "COUNT:"
!find outputs/videos_web/original_deepsort outputs/videos_web/best_yolo11n_osnet_x0_5 -name "*.mp4" | wc -l

!ffprobe -v error \
  -show_entries format=duration,size \
  -of default=noprint_wrappers=1 \
  outputs/videos_web/best_yolo11n_osnet_x0_5/TUD-Campus_best_yolo11n_osnet_x0_5.mp4

/content/deepsort-modern-project
COUNT:
12
duration=2.840000
size=270362


## 16. Verify browser-compatible videos again


In [27]:
%cd /content/deepsort-modern-project

!echo "COUNT:"
!find outputs/videos_web/original_deepsort outputs/videos_web/best_yolo11n_osnet_x0_5 -name "*.mp4" | wc -l

!echo ""
!echo "ORIGINAL WEB:"
!ls -lh outputs/videos_web/original_deepsort/*.mp4

!echo ""
!echo "BEST WEB:"
!ls -lh outputs/videos_web/best_yolo11n_osnet_x0_5/*.mp4

/content/deepsort-modern-project
COUNT:
12

ORIGINAL WEB:
-rw-r--r-- 1 root root 1.2M Jun 26 14:56 outputs/videos_web/original_deepsort/KITTI-17_original_deepsort.mp4
-rw-r--r-- 1 root root 7.8M Jun 26 14:57 outputs/videos_web/original_deepsort/MOT16-09_original_deepsort.mp4
-rw-r--r-- 1 root root  19M Jun 26 14:58 outputs/videos_web/original_deepsort/MOT16-11_original_deepsort.mp4
-rw-r--r-- 1 root root 4.2M Jun 26 14:58 outputs/videos_web/original_deepsort/PETS09-S2L1_original_deepsort.mp4
-rw-r--r-- 1 root root 258K Jun 26 14:58 outputs/videos_web/original_deepsort/TUD-Campus_original_deepsort.mp4
-rw-r--r-- 1 root root 545K Jun 26 14:58 outputs/videos_web/original_deepsort/TUD-Stadtmitte_original_deepsort.mp4

BEST WEB:
-rw-r--r-- 1 root root 1.2M Jun 26 14:58 outputs/videos_web/best_yolo11n_osnet_x0_5/KITTI-17_best_yolo11n_osnet_x0_5.mp4
-rw-r--r-- 1 root root 7.5M Jun 26 14:58 outputs/videos_web/best_yolo11n_osnet_x0_5/MOT16-09_best_yolo11n_osnet_x0_5.mp4
-rw-r--r-- 1 root root  

## 17. Display original overlay for TUD-Campus


In [30]:
%cd /content/deepsort-modern-project

from IPython.display import Video, display

print("Original DeepSORT overlay: TUD-Campus")
display(Video(
    "outputs/videos_web/original_deepsort/TUD-Campus_original_deepsort.mp4",
    embed=True,
    width=700
))

/content/deepsort-modern-project
Original DeepSORT overlay: TUD-Campus


## 18. Display best modified overlay for TUD-Campus


In [29]:
%cd /content/deepsort-modern-project

from IPython.display import Video, display

print("Best modified DeepSORT overlay: TUD-Campus")
display(Video(
    "outputs/videos_web/best_yolo11n_osnet_x0_5/TUD-Campus_best_yolo11n_osnet_x0_5.mp4",
    embed=True,
    width=700
))

/content/deepsort-modern-project
Best modified DeepSORT overlay: TUD-Campus


## 19. Save generated artifacts locally


In [31]:
%cd /content/deepsort-modern-project

import os
import shutil
import subprocess
from pathlib import Path
from datetime import datetime

PROJECT = Path("/content/deepsort-modern-project")
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

ART = PROJECT / "artifacts" / f"final_submission_{STAMP}"
ART.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("Saving important generated artifacts")
print("Project:", PROJECT)
print("Artifact folder:", ART)

small_dirs = [
    "outputs/tracks",
    "outputs/detections",
    "report",
]

for rel in small_dirs:
    src = PROJECT / rel
    dst = ART / rel
    if src.exists():
        print(f"Copying {src} -> {dst}")
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        print(f"Missing, skipped: {src}")

for rel in ["README.md"]:
    src = PROJECT / rel
    if src.exists():
        shutil.copy2(src, ART / rel)

manifest = ART / "MANIFEST.txt"
with open(manifest, "w") as f:
    f.write("DeepSORT Modernization Project - final generated artifacts\n")
    f.write(f"Saved at: {STAMP}\n\n")
    f.write("Included small artifacts:\n")
    f.write("- outputs/tracks: original and best tracking txt files\n")
    f.write("- outputs/detections: detector txt files\n")
    f.write("- report: report markdown and CSV result tables\n")
    f.write("- README.md\n\n")
    f.write("Large generated artifacts are stored in separate ZIP archives:\n")
    f.write("- final_outputs_no_videos_*.zip\n")
    f.write("- final_outputs_with_videos_*.zip\n\n")
    f.write("Expected final overlay count:\n")
    f.write("- 6 original overlay videos\n")
    f.write("- 6 best modified overlay videos\n")
    f.write("- total 12 overlay videos\n\n")

    f.write("Current files in outputs/tracks:\n")
    if (PROJECT / "outputs/tracks").exists():
        for p in sorted((PROJECT / "outputs/tracks").rglob("*.txt")):
            f.write(f"{p.relative_to(PROJECT)}\n")

    f.write("\nCurrent files in outputs/videos_web:\n")
    if (PROJECT / "outputs/videos_web").exists():
        for p in sorted((PROJECT / "outputs/videos_web").rglob("*.mp4")):
            f.write(f"{p.relative_to(PROJECT)}\n")

print("Manifest written:", manifest)

print("\n" + "=" * 80)
print("Current tracks:")
subprocess.run("find outputs/tracks -name '*.txt' -type f | sort | xargs -r wc -l", shell=True)

print("\nCurrent videos:")
subprocess.run("find outputs/videos outputs/videos_web -name '*.mp4' -type f | sort", shell=True)

print("\nVideo count:")
subprocess.run("find outputs/videos_web -name '*.mp4' -type f | wc -l", shell=True)

zip_no_videos = PROJECT / f"final_outputs_no_videos_{STAMP}"
items_no_videos = []
for rel in ["outputs/tracks", "outputs/detections", "outputs/detections_reid", "report", "README.md", str(ART.relative_to(PROJECT))]:
    if (PROJECT / rel).exists():
        items_no_videos.append(rel)

if items_no_videos:
    cmd = ["zip", "-r", "-q", str(zip_no_videos) + ".zip"] + items_no_videos
    print("\nCreating ZIP without videos:")
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)
    print("Created:", str(zip_no_videos) + ".zip")
else:
    print("No items for no-video ZIP")

zip_with_videos = PROJECT / f"final_outputs_with_videos_{STAMP}"
items_with_videos = []
for rel in ["outputs/tracks", "outputs/detections", "outputs/detections_reid", "outputs/videos_web", "report", "README.md", str(ART.relative_to(PROJECT))]:
    if (PROJECT / rel).exists():
        items_with_videos.append(rel)

if items_with_videos:
    cmd = ["zip", "-r", "-q", str(zip_with_videos) + ".zip"] + items_with_videos
    print("\nCreating ZIP with videos:")
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)
    print("Created:", str(zip_with_videos) + ".zip")
else:
    print("No items for video ZIP")

drive_dir = Path("/content/drive/MyDrive/deepsort_submission_backup")
if Path("/content/drive/MyDrive").exists():
    drive_dir.mkdir(parents=True, exist_ok=True)
    for z in [str(zip_no_videos) + ".zip", str(zip_with_videos) + ".zip"]:
        zp = Path(z)
        if zp.exists():
            dst = drive_dir / zp.name
            shutil.copy2(zp, dst)
            print("Copied to Drive:", dst)
else:
    print("Google Drive is not mounted. ZIPs remain in /content/deepsort-modern-project")

print("\n" + "=" * 80)
print("Final ZIP files:")
subprocess.run(f"ls -lh {PROJECT}/final_outputs_*_{STAMP}.zip", shell=True)

if drive_dir.exists():
    print("\nDrive backup folder:")
    subprocess.run(f"ls -lh {drive_dir}", shell=True)

print("\nDONE ")
print("Small artifacts folder:", ART)
print("No-video ZIP:", str(zip_no_videos) + ".zip")
print("With-video ZIP:", str(zip_with_videos) + ".zip")

/content/deepsort-modern-project
Saving important generated artifacts
Project: /content/deepsort-modern-project
Artifact folder: /content/deepsort-modern-project/artifacts/final_submission_20260626_150840
Copying /content/deepsort-modern-project/outputs/tracks -> /content/deepsort-modern-project/artifacts/final_submission_20260626_150840/outputs/tracks
Copying /content/deepsort-modern-project/outputs/detections -> /content/deepsort-modern-project/artifacts/final_submission_20260626_150840/outputs/detections
Copying /content/deepsort-modern-project/report -> /content/deepsort-modern-project/artifacts/final_submission_20260626_150840/report
Manifest written: /content/deepsort-modern-project/artifacts/final_submission_20260626_150840/MANIFEST.txt

Current tracks:

Current videos:

Video count:

Creating ZIP without videos:
zip -r -q /content/deepsort-modern-project/final_outputs_no_videos_20260626_150840.zip outputs/tracks outputs/detections outputs/detections_reid report README.md artifa

## 20. Optional: copy generated backup archives to Google Drive


In [32]:
from google.colab import drive
from pathlib import Path
import shutil
import subprocess

drive.mount("/content/drive", force_remount=True)

PROJECT = Path("/content/deepsort-modern-project")
BACKUP_DIR = Path("/content/drive/MyDrive/deepsort_submission_backup")
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("Checking generated files before backup")

print("\nTracks:")
subprocess.run("find /content/deepsort-modern-project/outputs/tracks -name '*.txt' -type f | sort | wc -l", shell=True)

print("\nWeb videos:")
subprocess.run("find /content/deepsort-modern-project/outputs/videos_web -name '*.mp4' -type f | sort | wc -l", shell=True)

print("\nZIPs in project:")
subprocess.run("ls -lh /content/deepsort-modern-project/final_outputs_*.zip", shell=True)

print("\nCopying ZIPs to Google Drive backup folder...")
for z in PROJECT.glob("final_outputs_*.zip"):
    dst = BACKUP_DIR / z.name
    shutil.copy2(z, dst)
    print("Copied:", dst)

print("\nBackup folder content:")
subprocess.run(f"ls -lh {BACKUP_DIR}", shell=True)

print("\nDONE Backup saved to:")
print(BACKUP_DIR)

Mounted at /content/drive
Checking generated files before backup

Tracks:

Web videos:

ZIPs in project:

Copying ZIPs to Google Drive backup folder...
Copied: /content/drive/MyDrive/deepsort_submission_backup/final_outputs_no_videos_20260626_150840.zip
Copied: /content/drive/MyDrive/deepsort_submission_backup/final_outputs_with_videos_20260626_150840.zip

Backup folder content:

DONE ✅ Backup saved to:
/content/drive/MyDrive/deepsort_submission_backup


## 21. Verify backup archive contents


In [33]:
from pathlib import Path
import zipfile

BACKUP_DIR = Path("/content/drive/MyDrive/deepsort_submission_backup")

zips = sorted(BACKUP_DIR.glob("final_outputs_*.zip"))

print("ZIP files in Drive:")
for z in zips:
    print(z.name, round(z.stat().st_size / 1024 / 1024, 2), "MB")

print("\nArchive content check:")
for z in zips:
    print("=" * 80)
    print(z.name)

    with zipfile.ZipFile(z, "r") as archive:
        names = archive.namelist()

        tracks = [n for n in names if n.startswith("outputs/tracks/") and n.endswith(".txt")]
        detections = [n for n in names if n.startswith("outputs/detections/") and n.endswith(".txt")]
        reid = [n for n in names if n.startswith("outputs/detections_reid/") and n.endswith(".npy")]
        videos = [n for n in names if n.startswith("outputs/videos_web/") and n.endswith(".mp4")]
        reports = [n for n in names if n.startswith("report/")]

        print("tracks txt:", len(tracks))
        print("detections txt:", len(detections))
        print("reid npy:", len(reid))
        print("web videos mp4:", len(videos))
        print("report files:", len(reports))

        if videos:
            print("video examples:")
            for n in videos[:4]:
                print(" ", n)

ZIP files in Drive:
final_outputs_no_videos_20260626_150840.zip 22.65 MB
final_outputs_with_videos_20260626_150840.zip 86.62 MB

Archive content check:
final_outputs_no_videos_20260626_150840.zip
tracks txt: 12
detections txt: 6
reid npy: 6
web videos mp4: 0
report files: 21
final_outputs_with_videos_20260626_150840.zip
tracks txt: 12
detections txt: 6
reid npy: 6
web videos mp4: 12
report files: 21
video examples:
  outputs/videos_web/best_yolo11n_osnet_x0_5/TUD-Stadtmitte_best_yolo11n_osnet_x0_5.mp4
  outputs/videos_web/best_yolo11n_osnet_x0_5/MOT16-11_best_yolo11n_osnet_x0_5.mp4
  outputs/videos_web/best_yolo11n_osnet_x0_5/PETS09-S2L1_best_yolo11n_osnet_x0_5.mp4
  outputs/videos_web/best_yolo11n_osnet_x0_5/TUD-Campus_best_yolo11n_osnet_x0_5.mp4
